# 1. Purpose

Validate the identity, schema, target, entity/time structure, quality, and leakage risks of NASA C-MAPSS FD001 before any modeling. This notebook is an audit only: it does not clean data, engineer features, fit models, or change raw files.

# 2. Imports and Configuration

In [1]:
from pathlib import Path
import hashlib
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "docs" / "data_strategy.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root")

REPO_ROOT = find_repository_root(Path.cwd().resolve())
RAW_DIR = REPO_ROOT / "data" / "raw" / "cmapss"
ARCHIVE_PATH = REPO_ROOT / "data" / "raw" / "CMAPSSData.zip"
TRAIN_PATH = RAW_DIR / "train_FD001.txt"
TEST_PATH = RAW_DIR / "test_FD001.txt"
TEST_RUL_PATH = RAW_DIR / "RUL_FD001.txt"

COLUMN_NAMES = (
    ["unit_number", "cycle"]
    + [f"operational_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

print(f"Repository root located: {REPO_ROOT.name}")

Repository root located: industrial-predictive-maintenance-decision-intelligence


# 3. Load Raw Data

Whitespace-delimited source files are read without modifying them. Column names follow the supplied C-MAPSS documentation: unit, cycle, three operational settings, and 21 sensor channels.

In [2]:
train = pd.read_csv(TRAIN_PATH, sep=r"\s+", header=None, names=COLUMN_NAMES)
test = pd.read_csv(TEST_PATH, sep=r"\s+", header=None, names=COLUMN_NAMES)
test_rul = pd.read_csv(
    TEST_RUL_PATH, sep=r"\s+", header=None, names=["rul_at_last_observation"]
)

{"train": train.shape, "test": test.shape, "test_rul": test_rul.shape}

{'train': (20631, 26), 'test': (13096, 26), 'test_rul': (100, 1)}

# 4. Dataset Identity Check

In [3]:
expected_archive_members = {
    "readme.txt", "Damage Propagation Modeling.pdf",
    *{f"train_FD00{i}.txt" for i in range(1, 5)},
    *{f"test_FD00{i}.txt" for i in range(1, 5)},
    *{f"RUL_FD00{i}.txt" for i in range(1, 5)},
}
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive_integrity_error = archive.testzip()
    archive_members = set(archive.namelist())

archive_sha256 = hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
identity_checks = pd.Series({
    "archive_integrity": archive_integrity_error is None,
    "expected_manifest": archive_members == expected_archive_members,
    "train_shape_20631_by_26": train.shape == (20631, 26),
    "test_shape_13096_by_26": test.shape == (13096, 26),
    "test_target_shape_100_by_1": test_rul.shape == (100, 1),
    "train_engines_100": train["unit_number"].nunique() == 100,
    "test_engines_100": test["unit_number"].nunique() == 100,
}, name="passed")
dataset_identity_status = "PASS" if identity_checks.all() else "WARNING"
display(identity_checks.to_frame())
print(f"Dataset identity check: {dataset_identity_status}")
print(f"Archive size: {ARCHIVE_PATH.stat().st_size:,} bytes")
print(f"Archive SHA-256: {archive_sha256}")

,passed
archive_integrity,True
expected_manifest,True
train_shape_20631_by_26,True
test_shape_13096_by_26,True
test_target_shape_100_by_1,True
train_engines_100,True
test_engines_100,True


Dataset identity check: PASS
Archive size: 12,425,978 bytes
Archive SHA-256: 74bef434a34db25c7bf72e668ea4cd52afe5f2cf8e44367c55a82bfd91a5a34f


# 5. Shape and Schema

In [4]:
display(train.head())
print(f"Training shape: {train.shape}")
print(f"Test shape: {test.shape}")
print("\nTraining frame info:")
train.info()
display(train.dtypes.rename("dtype").to_frame())
print("Columns:", train.columns.tolist())
print("Numeric columns:", train.select_dtypes(include="number").columns.tolist())
print("Categorical columns: []")
print("Identifier: unit_number; time/cycle: cycle; target: derived RUL")

,unit_number,cycle,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100,518.67,641.82,"1,589.7","1,400.6",14.62,21.61,554.36,"2,388.06","9,046.19",1.3,47.47,521.66,"2,388.02","8,138.62",8.4195,0.03,392,2388,100,39.06,23.419
1,1,2,0.0019,-0.0003,100,518.67,642.15,"1,591.82","1,403.14",14.62,21.61,553.75,"2,388.04","9,044.07",1.3,47.49,522.28,"2,388.07","8,131.49",8.4318,0.03,392,2388,100,39,23.4236
2,1,3,-0.0043,0.0003,100,518.67,642.35,"1,587.99","1,404.2",14.62,21.61,554.26,"2,388.08","9,052.94",1.3,47.27,522.42,"2,388.03","8,133.23",8.4178,0.03,390,2388,100,38.95,23.3442
3,1,4,0.0007,0,100,518.67,642.35,"1,582.79","1,401.87",14.62,21.61,554.45,"2,388.11","9,049.48",1.3,47.13,522.86,"2,388.08","8,133.83",8.3682,0.03,392,2388,100,38.88,23.3739
4,1,5,-0.0019,-0.0002,100,518.67,642.37,"1,582.85","1,406.22",14.62,21.61,554,"2,388.06","9,055.15",1.3,47.28,522.19,"2,388.04","8,133.8",8.4294,0.03,393,2388,100,38.9,23.4044


Training shape: (20631, 26)
Test shape: (13096, 26)

Training frame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   unit_number            20631 non-null  int64  
 1   cycle                  20631 non-null  int64  
 2   operational_setting_1  20631 non-null  float64
 3   operational_setting_2  20631 non-null  float64
 4   operational_setting_3  20631 non-null  float64
 5   sensor_1               20631 non-null  float64
 6   sensor_2               20631 non-null  float64
 7   sensor_3               20631 non-null  float64
 8   sensor_4               20631 non-null  float64
 9   sensor_5               20631 non-null  float64
 10  sensor_6               20631 non-null  float64
 11  sensor_7               20631 non-null  float64
 12  sensor_8               20631 non-null  float64
 13  sensor_9               20631 non

,dtype
unit_number,int64
cycle,int64
operational_setting_1,float64
operational_setting_2,float64
operational_setting_3,float64
sensor_1,float64
sensor_2,float64
sensor_3,float64
sensor_4,float64
sensor_5,float64


Columns: ['unit_number', 'cycle', 'operational_setting_1', 'operational_setting_2', 'operational_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']
Numeric columns: ['unit_number', 'cycle', 'operational_setting_1', 'operational_setting_2', 'operational_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']
Categorical columns: []
Identifier: unit_number; time/cycle: cycle; target: derived RUL


# 6. Missing Values

In [5]:
def missingness_summary(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.isna().sum().to_frame("missing_count")
    result["missing_pct"] = result["missing_count"] / len(frame)
    result["band"] = pd.cut(
        result["missing_pct"],
        bins=[-np.inf, 0, 0.05, 0.20, np.inf],
        labels=["0%", ">0-5%", "5-20%", ">20%"],
        include_lowest=True,
    )
    return result

train_missing = missingness_summary(train)
test_missing = missingness_summary(test)
display(train_missing)
print(f"Training missing cells: {train_missing['missing_count'].sum():,}")
print(f"Test missing cells: {test_missing['missing_count'].sum():,}")
print(f"Test-target missing cells: {test_rul.isna().sum().sum():,}")

,missing_count,missing_pct,band
unit_number,0,0,0%
cycle,0,0,0%
operational_setting_1,0,0,0%
operational_setting_2,0,0,0%
operational_setting_3,0,0,0%
sensor_1,0,0,0%
sensor_2,0,0,0%
sensor_3,0,0,0%
sensor_4,0,0,0%
sensor_5,0,0,0%


Training missing cells: 0
Test missing cells: 0
Test-target missing cells: 0


# 7. Duplicate Records

In [6]:
duplicate_summary = pd.DataFrame({
    "dataset": ["train", "test"],
    "duplicate_full_rows": [train.duplicated().sum(), test.duplicated().sum()],
    "duplicate_entity_cycle_keys": [
        train.duplicated(["unit_number", "cycle"]).sum(),
        test.duplicated(["unit_number", "cycle"]).sum(),
    ],
})
display(duplicate_summary)
print("Repeated numeric values in the one-column test RUL vector are valid targets for different engines and are not duplicate records.")

,dataset,duplicate_full_rows,duplicate_entity_cycle_keys
0,train,0,0
1,test,0,0


Repeated numeric values in the one-column test RUL vector are valid targets for different engines and are not duplicate records.


# 8. Target Distribution

FD001 uses a continuous RUL target. Binary positive/negative prevalence is therefore not applicable until the business horizon `H` is approved. Terminal `RUL = 0` rows are reported only as a descriptive endpoint count.

In [7]:
train_max_cycle = train.groupby("unit_number")["cycle"].transform("max")
train_audit = train.assign(rul=train_max_cycle - train["cycle"])
endpoint_count = int(train_audit["rul"].eq(0).sum())
endpoint_pct = train_audit["rul"].eq(0).mean()

target_summary = pd.DataFrame({
    "training_rul_all_cycles": train_audit["rul"].describe(),
    "test_rul_at_last_observation": test_rul["rul_at_last_observation"].describe(),
})
display(target_summary)
print("Binary target prevalence: N/A (H is not yet defined).")
print(f"Training terminal rows (RUL = 0): {endpoint_count:,} / {len(train_audit):,} ({endpoint_pct:.4%})")
print(f"Test RUL rows match unique test engines: {len(test_rul) == test['unit_number'].nunique()}")

,training_rul_all_cycles,test_rul_at_last_observation
count,"20,631",100
mean,107.808,75.52
std,68.881,41.765
min,0,7
25%,51,32.75
50%,103,86
75%,155,112.25
max,361,145


Binary target prevalence: N/A (H is not yet defined).
Training terminal rows (RUL = 0): 100 / 20,631 (0.4847%)
Test RUL rows match unique test engines: True


# 9. Numeric Range Checks

In [8]:
numeric_cols = train.select_dtypes(include="number").columns.tolist()
numeric_summary = train[numeric_cols].describe().T
numeric_summary["nunique"] = train[numeric_cols].nunique()
numeric_summary["zero_count"] = train[numeric_cols].eq(0).sum()
numeric_summary["negative_count"] = train[numeric_cols].lt(0).sum()
numeric_summary["nonfinite_count"] = np.isinf(train[numeric_cols]).sum()
display(numeric_summary)

constant_columns = numeric_summary.index[numeric_summary["nunique"].eq(1)].tolist()
dominant_share = train[numeric_cols].apply(lambda series: series.value_counts(normalize=True).iloc[0])
near_constant_columns = dominant_share[(dominant_share.ge(0.95)) & (~dominant_share.index.isin(constant_columns))].index.tolist()
print("Constant columns:", constant_columns)
print("Near-constant columns (>=95% one value):", near_constant_columns)

range_exceedance = []
for column in COLUMN_NAMES[2:]:
    below = int(test[column].lt(train[column].min()).sum())
    above = int(test[column].gt(train[column].max()).sum())
    if below or above:
        range_exceedance.append({"column": column, "below_train_min": below, "above_train_max": above})
display(pd.DataFrame(range_exceedance))

range_findings = pd.DataFrame([
    {"finding": "Seven constant raw columns", "classification": "NEEDS DOMAIN REVIEW", "action": "Mark REMOVE; do not alter raw data"},
    {"finding": "sensor_6 has two values and 98.03% at 21.61", "classification": "NEEDS DOMAIN REVIEW", "action": "Mark REMOVE provisionally"},
    {"finding": "Small negative operational settings", "classification": "PLAUSIBLE EXTREME", "action": "Retain; settings appear encoded but units need verification"},
    {"finding": "A few test values exceed training ranges", "classification": "PLAUSIBLE EXTREME", "action": "Retain and monitor; never clip automatically"},
    {"finding": "No NaN or infinite numeric values", "classification": "VALID", "action": "No remediation"},
])
display(range_findings)

,count,mean,std,min,25%,50%,75%,max,nunique,zero_count,negative_count,nonfinite_count
unit_number,"20,631",51.5066,29.2276,1,26,52,77,100,100,0,0,0
cycle,"20,631",108.808,68.881,1,52,104,156,362,362,0,0,0
operational_setting_1,"20,631",-8.87015e-06,0.00218731,-0.0087,-0.0015,0,0.0015,0.0087,158,413,10061,0
operational_setting_2,"20,631",2.35083e-06,0.000293062,-0.0006,-0.0002,0,0.0003,0.0006,13,2070,9225,0
operational_setting_3,"20,631",100,0,100,100,100,100,100,1,0,0,0
sensor_1,"20,631",518.67,0,518.67,518.67,518.67,518.67,518.67,1,0,0,0
sensor_2,"20,631",642.681,0.500053,641.21,642.325,642.64,643,644.53,310,0,0,0
sensor_3,"20,631","1,590.52",6.13115,"1,571.04","1,586.26","1,590.1","1,594.38","1,616.91",3012,0,0,0
sensor_4,"20,631","1,408.93",9.0006,"1,382.25","1,402.36","1,408.04","1,414.55","1,441.49",4051,0,0,0
sensor_5,"20,631",14.62,1.7764e-15,14.62,14.62,14.62,14.62,14.62,1,0,0,0


Constant columns: ['operational_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Near-constant columns (>=95% one value): ['sensor_6']


,column,below_train_min,above_train_max
0,operational_setting_2,0,5
1,sensor_2,2,0
2,sensor_3,3,0
3,sensor_8,1,0
4,sensor_11,2,0
5,sensor_12,0,3
6,sensor_21,0,2


,finding,classification,action
0,Seven constant raw columns,NEEDS DOMAIN REVIEW,Mark REMOVE; do not alter raw data
1,sensor_6 has two values and 98.03% at 21.61,NEEDS DOMAIN REVIEW,Mark REMOVE provisionally
2,Small negative operational settings,PLAUSIBLE EXTREME,Retain; settings appear encoded but units need...
3,A few test values exceed training ranges,PLAUSIBLE EXTREME,Retain and monitor; never clip automatically
4,No NaN or infinite numeric values,VALID,No remediation


# 10. Categorical Value Checks

In [9]:
categorical_cols = train.select_dtypes(exclude="number").columns.tolist()
print("Categorical columns:", categorical_cols)
print("No categorical-label, whitespace, case, or rare-category checks apply because all 26 raw fields are numeric.")

Categorical columns: []
No categorical-label, whitespace, case, or rare-category checks apply because all 26 raw fields are numeric.


# 11. Entity and Time Structure

In [10]:
def entity_time_summary(frame: pd.DataFrame, name: str) -> dict:
    rows_per_engine = frame.groupby("unit_number").size()
    monotonic = frame.groupby("unit_number")["cycle"].apply(lambda values: values.is_monotonic_increasing).all()
    consecutive = frame.groupby("unit_number")["cycle"].apply(
        lambda values: values.tolist() == list(range(1, len(values) + 1))
    ).all()
    return {
        "dataset": name,
        "unique_engines": frame["unit_number"].nunique(),
        "rows_per_engine_mean": rows_per_engine.mean(),
        "rows_per_engine_min": rows_per_engine.min(),
        "rows_per_engine_median": rows_per_engine.median(),
        "rows_per_engine_max": rows_per_engine.max(),
        "cycle_min": frame["cycle"].min(),
        "cycle_max": frame["cycle"].max(),
        "all_monotonic": monotonic,
        "all_consecutive_from_1": consecutive,
        "duplicate_engine_cycle_keys": frame.duplicated(["unit_number", "cycle"]).sum(),
    }

entity_summary = pd.DataFrame([entity_time_summary(train, "train"), entity_time_summary(test, "test")])
display(entity_summary)
print("Validation implication: split by engine before creating samples or windows; random row-level splitting is prohibited.")

,dataset,unique_engines,rows_per_engine_mean,rows_per_engine_min,rows_per_engine_median,rows_per_engine_max,cycle_min,cycle_max,all_monotonic,all_consecutive_from_1,duplicate_engine_cycle_keys
0,train,100,206.31,128,199,362,1,362,True,True,0
1,test,100,130.96,31,133.5,303,1,303,True,True,0


Validation implication: split by engine before creating samples or windows; random row-level splitting is prohibited.


# 12. Leakage Audit

In [11]:
remove_columns = {"operational_setting_3", "sensor_1", "sensor_5", "sensor_6", "sensor_10", "sensor_16", "sensor_18", "sensor_19"}
keep_columns = {"cycle", "operational_setting_1", "operational_setting_2"}

leakage_rows = []
for column in COLUMN_NAMES:
    if column == "unit_number":
        leakage_rows.append({"feature": column, "available_before_decision": "Yes", "leakage_risk": "MEDIUM", "decision": "ID ONLY", "reason": "Could memorize unit-specific lifetime; grouping and split key only"})
    elif column == "cycle":
        leakage_rows.append({"feature": column, "available_before_decision": "Yes", "leakage_risk": "MEDIUM", "decision": "KEEP", "reason": "Known age, but may exploit benchmark lifetime distribution"})
    elif column in remove_columns:
        leakage_rows.append({"feature": column, "available_before_decision": "Yes", "leakage_risk": "LOW", "decision": "REMOVE", "reason": "Constant or near-constant in FD001; quality issue rather than target leakage"})
    elif column in keep_columns:
        leakage_rows.append({"feature": column, "available_before_decision": "Yes", "leakage_risk": "LOW", "decision": "KEEP", "reason": "Contemporaneous operating context"})
    else:
        leakage_rows.append({"feature": column, "available_before_decision": "Yes", "leakage_risk": "LOW", "decision": "INVESTIGATE", "reason": "Contemporaneous sensor, but physical meaning and units are not documented"})

leakage_rows.extend([
    {"feature": "rul / official test RUL", "available_before_decision": "No", "leakage_risk": "HIGH", "decision": "TARGET", "reason": "Direct outcome; evaluation use only"},
    {"feature": "T_failure / per-unit maximum cycle", "available_before_decision": "No", "leakage_risk": "HIGH", "decision": "REMOVE", "reason": "Post-outcome value used only to construct training labels"},
    {"feature": "future sensor rows or future-containing aggregates", "available_before_decision": "No", "leakage_risk": "HIGH", "decision": "REMOVE", "reason": "Contains information after prediction cycle t"},
    {"feature": "same engine across development splits", "available_before_decision": "N/A", "leakage_risk": "HIGH", "decision": "REMOVE", "reason": "Shares identity and degradation history across train and validation"},
])
leakage_audit = pd.DataFrame(leakage_rows)
display(leakage_audit)

,feature,available_before_decision,leakage_risk,decision,reason
0,unit_number,Yes,MEDIUM,ID ONLY,Could memorize unit-specific lifetime; groupin...
1,cycle,Yes,MEDIUM,KEEP,"Known age, but may exploit benchmark lifetime ..."
2,operational_setting_1,Yes,LOW,KEEP,Contemporaneous operating context
3,operational_setting_2,Yes,LOW,KEEP,Contemporaneous operating context
4,operational_setting_3,Yes,LOW,REMOVE,Constant or near-constant in FD001; quality is...
5,sensor_1,Yes,LOW,REMOVE,Constant or near-constant in FD001; quality is...
6,sensor_2,Yes,LOW,INVESTIGATE,"Contemporaneous sensor, but physical meaning a..."
7,sensor_3,Yes,LOW,INVESTIGATE,"Contemporaneous sensor, but physical meaning a..."
8,sensor_4,Yes,LOW,INVESTIGATE,"Contemporaneous sensor, but physical meaning a..."
9,sensor_5,Yes,LOW,REMOVE,Constant or near-constant in FD001; quality is...


# 13. Data Quality Findings

In [12]:
quality_findings = pd.DataFrame([
    {"issue": "Dataset identity", "severity": "LOW", "evidence": "Archive integrity, manifest, dimensions, and engine counts pass", "modeling_impact": "No identity blocker", "recommended_action": "Retain checksum as local baseline"},
    {"issue": "Missing or non-finite values", "severity": "LOW", "evidence": "0 missing and 0 infinite cells", "modeling_impact": "No imputation currently indicated", "recommended_action": "Recheck in reproducible intake validation"},
    {"issue": "Duplicate records and keys", "severity": "LOW", "evidence": "0 duplicate rows and 0 duplicate engine-cycle keys", "modeling_impact": "No deduplication indicated", "recommended_action": "Retain immutable raw rows"},
    {"issue": "Constant and near-constant fields", "severity": "MEDIUM", "evidence": "7 constant columns; sensor_6 is 98.03% one value", "modeling_impact": "No useful signal and possible numerical noise", "recommended_action": "Exclude the 8 fields from future predictors"},
    {"issue": "Anonymized sensor semantics", "severity": "HIGH", "evidence": "Physical names and units absent from supplied README", "modeling_impact": "Limits physical range validation and business explanation", "recommended_action": "Verify from authoritative documentation; retain variable sensors as INVESTIGATE"},
    {"issue": "Test RUL ordering", "severity": "HIGH", "evidence": "100 targets align by count, but README does not state row-to-engine ordering", "modeling_impact": "Wrong mapping would invalidate final evaluation", "recommended_action": "Confirm authoritative mapping before using official test labels"},
    {"issue": "Undefined business horizon H", "severity": "MEDIUM", "evidence": "No mapping from cycles to maintenance window", "modeling_impact": "Cannot define binary prevalence or economic action label", "recommended_action": "Approve H before classification or economic evaluation"},
    {"issue": "Synthetic and narrow FD001 scope", "severity": "HIGH", "evidence": "Simulated turbofans; one condition and one fault mode", "modeling_impact": "Limited external validity for a manufacturing fleet", "recommended_action": "Frame results as benchmark feasibility only"},
    {"issue": "Small train-range exceedances in test", "severity": "LOW", "evidence": "Seven fields have only 1-5 out-of-range test rows", "modeling_impact": "Minor deployment-range warning", "recommended_action": "Monitor; do not clip automatically"},
])
display(quality_findings)

,issue,severity,evidence,modeling_impact,recommended_action
0,Dataset identity,LOW,"Archive integrity, manifest, dimensions, and e...",No identity blocker,Retain checksum as local baseline
1,Missing or non-finite values,LOW,0 missing and 0 infinite cells,No imputation currently indicated,Recheck in reproducible intake validation
2,Duplicate records and keys,LOW,0 duplicate rows and 0 duplicate engine-cycle ...,No deduplication indicated,Retain immutable raw rows
3,Constant and near-constant fields,MEDIUM,7 constant columns; sensor_6 is 98.03% one value,No useful signal and possible numerical noise,Exclude the 8 fields from future predictors
4,Anonymized sensor semantics,HIGH,Physical names and units absent from supplied ...,Limits physical range validation and business ...,Verify from authoritative documentation; retai...
5,Test RUL ordering,HIGH,"100 targets align by count, but README does no...",Wrong mapping would invalidate final evaluation,Confirm authoritative mapping before using off...
6,Undefined business horizon H,MEDIUM,No mapping from cycles to maintenance window,Cannot define binary prevalence or economic ac...,Approve H before classification or economic ev...
7,Synthetic and narrow FD001 scope,HIGH,Simulated turbofans; one condition and one fau...,Limited external validity for a manufacturing ...,Frame results as benchmark feasibility only
8,Small train-range exceedances in test,LOW,Seven fields have only 1-5 out-of-range test rows,Minor deployment-range warning,Monitor; do not clip automatically


# 14. Modeling Readiness Decision

In [13]:
MODELING_READINESS = "CONDITIONALLY READY"
print(MODELING_READINESS)
print("\nTop 3 reasons:")
print("1. Identity, completeness, uniqueness, and engine-cycle ordering checks pass.")
print("2. The RUL target and engine-disjoint validation structure are feasible.")
print("3. Constant fields, anonymized sensor semantics, test-target ordering, and horizon H still require controls or clarification.")
print("\nRequired fixes before modeling:")
print("- Freeze engine-disjoint development splits before any fitted transformation or window construction.")
print("- Exclude unit_number, seven constant fields, near-constant sensor_6, and all target/post-outcome information from predictors.")
print("- Confirm test RUL row-to-engine ordering and document anonymized sensor limitations.")
print("- Approve horizon H before creating a binary failure-within-window target or linking predictions to economics.")

CONDITIONALLY READY

Top 3 reasons:
1. Identity, completeness, uniqueness, and engine-cycle ordering checks pass.
2. The RUL target and engine-disjoint validation structure are feasible.
3. Constant fields, anonymized sensor semantics, test-target ordering, and horizon H still require controls or clarification.

Required fixes before modeling:
- Freeze engine-disjoint development splits before any fitted transformation or window construction.
- Exclude unit_number, seven constant fields, near-constant sensor_6, and all target/post-outcome information from predictors.
- Confirm test RUL row-to-engine ordering and document anonymized sensor limitations.
- Approve horizon H before creating a binary failure-within-window target or linking predictions to economics.
